In [1]:
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import scipy.io as sio
import os
import liana as li
from liana.method import cellphonedb, geometric_mean

# seed for reproducibility
RNG_SEED = 1337
np.random.seed(RNG_SEED) 

export_dir = "output/03a_RCTD_spotdeconvolution/ref_export"   # written by 03a_RCTD_spotdeconvolution.qmd
out_dir    = "output/03b_runLiana_scRNAseq"
os.makedirs(out_dir, exist_ok=True)

counts = sio.mmread(f"{export_dir}/lognormalized_matrix.mtx").tocsr().T
genes = pd.read_csv(f"{export_dir}/features.tsv", header=None)[0].values
barcodes = pd.read_csv(f"{export_dir}/barcodes.tsv", header=None)[0].values
meta = pd.read_csv(f"{export_dir}/metadata.csv").set_index("barcode").loc[barcodes]

adata = ad.AnnData(X=counts, obs=meta, var=pd.DataFrame(index=genes))
adata.var_names_make_unique()
adata.raw = adata   # already log-normalized from R

# candidate_resource.csv pools the significant candidate LR pairs from all 4
# spatial pipelines (Astrocyte-Vascular, Vascular-Microglia, OPC-Cajal.Retzius,
# Neurogenic_lineage-GC) — see the R "Save data for LIANA+ CCC" section in 03a. Running
# the permutation test once on the pooled resource, then filtering per cell-type
# pair below, avoids paying the (expensive) permutation cost 4 times.
candidate_resource = pd.read_csv(f"{export_dir}/candidate_resource.csv")

# "Celltype" is already ref_seurat$Celltype_updated from 03a (qNSC/nIPC/Neuroblast ->
# Neurogenic_lineage, SMC/Pericyte/Endothelial -> Vascular, Cajal-Retzius -> Cajal.Retzius),
# so LIANA's groups match the RCTD cell types used throughout the spatial pipeline
adata.obs["Celltype"].value_counts()  


methods = [cellphonedb, geometric_mean]
custom_aggregate = li.mt.AggregateClass(li.mt.aggregate_meta, methods=methods)

results = []
for age_label in ["Young", "Middle", "Old"]:
    sub = adata[adata.obs["Age"] == age_label].copy()

    custom_aggregate(sub,
                      groupby="Celltype",
                      resource=candidate_resource,
                      expr_prop=0.1,
                      n_perms=1000,
                      seed=RNG_SEED,
                      verbose=True,
                      key_added="cpdb_gmean_agg")

    res_age = sub.uns["cpdb_gmean_agg"].copy()
    res_age["Age"] = age_label
    results.append(res_age)

res_by_age = pd.concat(results, ignore_index=True)


/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning:

Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.

/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning:

Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.

/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning:

Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.

/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning:

Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.

/home/vkorob/miniconda3/envs/lianaPlus/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning:

Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.

/home/vkorob/miniconda3/envs/lia

Using `.raw`!


Converting to sparse csr matrix!


Using provided `resource`.


11004 features of mat are empty, they will be removed.


...

 98%|█████████▊| 985/1000 [00:21<00:00, 45.95it/s]

 99%|█████████▉| 990/1000 [00:21<00:00, 45.86it/s]

100%|█████████▉| 995/1000 [00:21<00:00, 46.39it/s]

100%|██████████| 1000/1000 [00:21<00:00, 46.41it/s]

100%|██████████| 1000/1000 [00:21<00:00, 46.45it/s]

In [2]:
# res_by_age already has every cell-type pair present in the data (groupby="Celltype"
# runs CCC across all pairs at once) — extract_pair() just filters the one shared
# result table per pair of interest, both interaction directions included.

def extract_pair(res_df, ct1, ct2):
    return res_df[
        ((res_df["source"] == ct1) & (res_df["target"] == ct2)) |
        ((res_df["source"] == ct2) & (res_df["target"] == ct1))
    ].copy()

def finalize_and_save(res_pair, filename):
    res_pair = res_pair.rename(columns={"Age": "age"})
    res_pair["age"] = res_pair["age"].str.lower()
    res_pair.to_csv(os.path.join(out_dir, filename), index=False)
    return res_pair

res_astro_vasc_by_age    = finalize_and_save(extract_pair(res_by_age, "Astrocyte", "Vascular"),
                                              "liana_astro_vasc_by_age.csv")
res_vasc_microglia_by_age = finalize_and_save(extract_pair(res_by_age, "Vascular", "Microglia"),
                                               "liana_vasc_microglia_by_age.csv")
res_opc_cr_by_age        = finalize_and_save(extract_pair(res_by_age, "OPC", "Cajal.Retzius"),
                                              "liana_opc_cr_by_age.csv")
res_neuro_gc_by_age      = finalize_and_save(extract_pair(res_by_age, "Neurogenic_lineage", "GC"),
                                              "liana_neuro_gc_by_age.csv")

# sanity check: each should have rows, both interaction directions, all 3 ages
for name, df in [("astro_vasc", res_astro_vasc_by_age),
                  ("vasc_microglia", res_vasc_microglia_by_age),
                  ("opc_cr", res_opc_cr_by_age),
                  ("neuro_gc", res_neuro_gc_by_age)]:
    print(name, "rows:", len(df), "| sources:", df["source"].unique(), "| ages:", df["age"].unique())


astro_vasc rows: 391 | sources: ['Vascular' 'Astrocyte'] | ages: ['young' 'middle' 'old']
vasc_microglia rows: 325 | sources: ['Vascular' 'Microglia'] | ages: ['young' 'middle' 'old']
opc_cr rows: 546 | sources: ['OPC' 'Cajal.Retzius'] | ages: ['young' 'middle' 'old']
neuro_gc rows: 855 | sources: ['Neurogenic_lineage' 'GC'] | ages: ['young' 'middle' 'old']
